# Analysis Suite — Deep Statistical Dive

This notebook performs a rigorous statistical comparison of four Monte Carlo pricing techniques:

| # | Technique | Key Idea |
|---|-----------|----------|
| 1 | **Standard MC** | Pseudo-random iid baseline |
| 2 | **Antithetic Variates** | Negative correlation via $Z$ and $-Z$ pairs |
| 3 | **Control Variates** | BS-anchored correction for Asian options |
| 4 | **Sobol QMC** | Low-discrepancy quasi-random sequences |

Analyses:
1. Log(Error) vs Log(N) convergence rates
2. 95 % Confidence Interval width comparison
3. Error distribution (histogram + Q-Q plot)
4. MC Greeks vs analytical BS Greeks
5. Variance Reduction & Efficiency Scorecard

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

from src.base_simulator import SimulatorConfig
from src.stochastic_processes import GBMSimulator
from src.payoffs import VanillaCall, AsianArithmetic
from src.analytical import black_scholes, compute_error_metrics, compute_greeks
from src.variance_reduction import control_variate_price, mc_greeks

plt.style.use('seaborn-v0_8-whitegrid')
FIGSIZE_WIDE = (14, 5)
FIGSIZE_SQ   = (7, 5)

# ── Baseline config (ATM, 1-year horizon) ─────────────────────────────────
BASE = SimulatorConfig(S0=100.0, K=100.0, T=1.0, r=0.05, sigma=0.2,
                       n_paths=50_000, n_steps=252, seed=None)

BS_CALL   = black_scholes(BASE, 'call')
BS_GREEKS = compute_greeks(BASE, 'call')

# ── Asian reference price (high-N MC, used for Sections 1–3) ──────────────
# Since no closed-form exists for arithmetic Asian calls, we use a very
# large simulation (200 k paths, fixed seed) as the ground-truth reference.
print('Computing Asian reference price (200k paths)...')
_ref_cfg  = SimulatorConfig(S0=BASE.S0, K=BASE.K, T=BASE.T, r=BASE.r,
                             sigma=BASE.sigma, n_paths=200_000, n_steps=252, seed=0)
ASIAN_REF = GBMSimulator(_ref_cfg).price(AsianArithmetic(BASE.K), 'standard')['price']

print(f'Black-Scholes call  : ${BS_CALL:.6f}')
print(f'Asian Ref (200k)    : ${ASIAN_REF:.6f}  (used as ground truth for Asian analyses)')
print(f'BS Delta            : {BS_GREEKS["delta"]:.6f}')
print(f'BS Vega (per 1%σ)   : {BS_GREEKS["vega"]:.6f}')


---
## Section 1 — Convergence Rate Analysis

All four methods price the **Asian Arithmetic Call** (the hardest option here).  
The reference price is `ASIAN_REF` — a stable 200k-path simulation.

Control Variates uses the European call (BS price known) as a control to correct  
the Asian call estimate — this is where CV earns its variance-reduction advantage.

The plot shows $\log_{10}(|\hat{V} - V_{\text{ref}}|)$ vs $\log_{10}(N)$:
- Standard MC slope $\approx -0.5$  →$O(1/\sqrt{N})$
- Antithetic: same slope, smaller constant  
- Control Variates / Sobol: slope approaches $-1.0$  →$O(1/N)$


In [ ]:
N_SWEEP       = [200, 500, 1_000, 2_000, 5_000, 10_000, 25_000, 50_000]
N_STEPS_SWEEP = 50   # fast — convergence is vs N, not steps
N_SEEDS       = 5    # independent runs per N to average out seed variance

METHODS = {
    'Standard MC':      dict(color='#1f77b4', ls='-'),
    'Antithetic':       dict(color='#ff7f0e', ls='--'),
    'Control Variates': dict(color='#2ca02c', ls='-.'),
    'Sobol QMC':        dict(color='#9467bd', ls=':'),
}

# Each entry: list of mean-over-seeds absolute errors per N value
errors = {m: [] for m in METHODS}

for n in N_SWEEP:
    errs_n = {m: [] for m in METHODS}
    for _ in range(N_SEEDS):
        cfg = SimulatorConfig(S0=BASE.S0, K=BASE.K, T=BASE.T, r=BASE.r,
                              sigma=BASE.sigma, n_paths=n,
                              n_steps=N_STEPS_SWEEP, seed=None)
        sim     = GBMSimulator(cfg)
        pay_asi = AsianArithmetic(cfg.K)
        pay_eur = VanillaCall(cfg.K)
        bs_ctrl = black_scholes(cfg, 'call')

        # All four methods price the Asian call; reference = ASIAN_REF
        errs_n['Standard MC'].append(
            abs(sim.price(pay_asi, 'standard')['price']   - ASIAN_REF))
        errs_n['Antithetic'].append(
            abs(sim.price(pay_asi, 'antithetic')['price'] - ASIAN_REF))
        errs_n['Sobol QMC'].append(
            abs(sim.price(pay_asi, 'sobol')['price']      - ASIAN_REF))
        cv = control_variate_price(sim, pay_asi, pay_eur, bs_ctrl)
        errs_n['Control Variates'].append(abs(cv['price'] - ASIAN_REF))

    for m in METHODS:
        errors[m].append(np.mean(errs_n[m]))

    print(f'N={n:>6,}  ' + '  '.join(
        f"{m[:4]}={errors[m][-1]:.4f}" for m in METHODS))

print('\nConvergence sweep done.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)

# Left: raw error vs N (log-log)
ax = axes[0]
log_N = np.log10(N_SWEEP)
for m, style in METHODS.items():
    log_e = np.log10(np.array(errors[m]) + 1e-8)
    slope, intercept = np.polyfit(log_N, log_e, 1)
    ax.plot(N_SWEEP, errors[m], marker='o', ms=5,
            color=style['color'], ls=style['ls'],
            label=f'{m}  (slope ≈ {slope:.2f})')

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Number of Paths  N', fontsize=12)
ax.set_ylabel('|MC Price − BS Price|', fontsize=12)
ax.set_title('Absolute Error vs N  (log-log scale)', fontsize=13)
ax.legend(fontsize=9)

# Right: fitted slopes as bar chart
ax2 = axes[1]
slopes = []
for m in METHODS:
    log_e = np.log10(np.array(errors[m]) + 1e-8)
    slope, _ = np.polyfit(log_N, log_e, 1)
    slopes.append(slope)

colors_list = [v['color'] for v in METHODS.values()]
bars = ax2.barh(list(METHODS.keys()), slopes, color=colors_list, alpha=0.85)
ax2.axvline(-0.5, color='black', ls='--', lw=1, label='O(1/√N) = −0.5')
ax2.axvline(-1.0, color='red',   ls='--', lw=1, label='O(1/N) = −1.0')
ax2.set_xlabel('Empirical convergence slope', fontsize=12)
ax2.set_title('Convergence Rate Comparison', fontsize=13)
ax2.legend(fontsize=9)
for bar, slope in zip(bars, slopes):
    ax2.text(slope - 0.02, bar.get_y() + bar.get_height()/2,
             f'{slope:.2f}', va='center', ha='right', fontsize=9, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('../notebooks/plot_convergence_rates.png', dpi=150)
plt.show()

---
## Section 2 — Confidence Interval Width

At a fixed $N = 50{,}000$, compare the 95 % CI width of all four methods  
**pricing the Asian Arithmetic Call**:

$$\text{CI width} = 2 \times 1.96 \times \hat{\sigma}_{\bar{X}}$$

A narrower CI means higher *precision* — the same accuracy is achievable with fewer paths.  
Control Variates should show a dramatically narrower CI than Standard MC.


In [ ]:
N_CI = 50_000
cfg_ci  = SimulatorConfig(S0=BASE.S0, K=BASE.K, T=BASE.T, r=BASE.r,
                          sigma=BASE.sigma, n_paths=N_CI, n_steps=252, seed=None)
sim_ci  = GBMSimulator(cfg_ci)
pay_asi = AsianArithmetic(cfg_ci.K)
pay_eur = VanillaCall(cfg_ci.K)
bs_ctrl = black_scholes(cfg_ci, 'call')

# All four methods price the Asian call for a fair comparison
ci_results = {
    'Standard MC':      sim_ci.price(pay_asi, 'standard'),
    'Antithetic':       sim_ci.price(pay_asi, 'antithetic'),
    'Control Variates': control_variate_price(sim_ci, pay_asi, pay_eur, bs_ctrl),
    'Sobol QMC':        sim_ci.price(pay_asi, 'sobol'),
}

ci_widths = {m: r['conf_interval'][1] - r['conf_interval'][0]
             for m, r in ci_results.items()}
std_w = ci_widths['Standard MC']

fig, ax = plt.subplots(figsize=(9, 5))
method_names = list(ci_widths.keys())
width_vals   = list(ci_widths.values())
bar_colors   = [METHODS[m]['color'] for m in method_names]

bars = ax.bar(method_names, width_vals, color=bar_colors, alpha=0.85, edgecolor='white')
for bar, w, name in zip(bars, width_vals, method_names):
    pct   = (1 - w / std_w) * 100
    label = (f'{w:.4f}\n(baseline)' if name == 'Standard MC'
             else f'{w:.4f}\n{pct:.0f}% narrower')
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + std_w * 0.01,
            label, ha='center', va='bottom', fontsize=9)

ax.set_ylabel('95% CI Width', fontsize=12)
ax.set_title(f'CI Width Comparison — Asian Arithmetic Call  (N = {N_CI:,})', fontsize=13)
plt.tight_layout()
plt.savefig('../notebooks/plot_ci_width.png', dpi=150)
plt.show()

for m, w in ci_widths.items():
    pct = (1 - w / std_w) * 100
    print(f'{m:>22s}: CI width = {w:.5f}  ({pct:+.1f}% vs Standard MC)')


---
## Section 3 — Error Distribution

Run each technique 200 times at $N = 1{,}000$ (all pricing the **Asian call**).  
Residual = $\hat{V}_{\text{Asian}} - V_{\text{ASIAN\_REF}}$.

- Histograms: show the spread and bias of each estimator.  
- Q-Q plots: test normality (CLT should hold for large N, but deviations  
  show up clearly at small N for QMC sequences).  
- The **standard deviation of residuals** is the square root of variance —  
  CV and Sobol should have dramatically smaller spread.


In [ ]:
N_DIST   = 1_000   # paths per run
N_RUNS   = 200     # independent runs

residuals = {m: [] for m in METHODS}

for _ in range(N_RUNS):
    cfg_d   = SimulatorConfig(S0=BASE.S0, K=BASE.K, T=BASE.T, r=BASE.r,
                              sigma=BASE.sigma, n_paths=N_DIST,
                              n_steps=N_STEPS_SWEEP, seed=None)
    sim_d   = GBMSimulator(cfg_d)
    pay_asi = AsianArithmetic(cfg_d.K)
    pay_eur = VanillaCall(cfg_d.K)
    bs_c    = black_scholes(cfg_d, 'call')

    # All four methods price the Asian call — residuals vs ASIAN_REF
    residuals['Standard MC'].append(
        sim_d.price(pay_asi, 'standard')['price']   - ASIAN_REF)
    residuals['Antithetic'].append(
        sim_d.price(pay_asi, 'antithetic')['price'] - ASIAN_REF)
    residuals['Sobol QMC'].append(
        sim_d.price(pay_asi, 'sobol')['price']      - ASIAN_REF)
    cv_r = control_variate_price(sim_d, pay_asi, pay_eur, bs_c)
    residuals['Control Variates'].append(cv_r['price']  - ASIAN_REF)

for m in METHODS:
    arr = np.array(residuals[m])
    print(f'{m:>22s}: mean={arr.mean():.4f}  std={arr.std():.4f}')


In [ ]:
fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

for col_idx, (m, style) in enumerate(METHODS.items()):
    arr = np.array(residuals[m])

    # Top row: Histogram
    ax_h = fig.add_subplot(gs[0, col_idx])
    ax_h.hist(arr, bins=30, color=style['color'], alpha=0.75, density=True,
              edgecolor='white')
    x_r = np.linspace(arr.min(), arr.max(), 200)
    ax_h.plot(x_r, stats.norm.pdf(x_r, arr.mean(), arr.std()),
              color='black', lw=1.5, ls='--', label='Normal fit')
    ax_h.axvline(0, color='red', lw=1, ls=':')
    ax_h.set_title(m, fontsize=10, fontweight='bold', color=style['color'])
    ax_h.set_xlabel('Residual  (MC − ASIAN_REF)', fontsize=8)
    ax_h.set_ylabel('Density', fontsize=8)
    ax_h.annotate(f'std={arr.std():.4f}', xy=(0.97, 0.92), xycoords='axes fraction',
                  ha='right', fontsize=8)

    # Bottom row: Q-Q plot
    ax_q = fig.add_subplot(gs[1, col_idx])
    (osm, osr), (slope, intercept, r) = stats.probplot(arr, dist='norm')
    ax_q.plot(osm, osr, 'o', ms=3, color=style['color'], alpha=0.6)
    ax_q.plot(osm, slope*np.array(osm)+intercept, 'k--', lw=1.5)
    ax_q.set_title(f'Q-Q  (R²={r**2:.3f})', fontsize=9)
    ax_q.set_xlabel('Theoretical quantiles', fontsize=8)
    ax_q.set_ylabel('Sample quantiles', fontsize=8)

fig.suptitle(
    f'Error Distributions — {N_RUNS} runs × N={N_DIST:,} paths  (Asian Arithmetic Call)\n'
    f'(reference price = ${ASIAN_REF:.4f}; zero-line marks perfect estimate)',
    fontsize=12, y=1.01
)
plt.savefig('../notebooks/plot_error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Section 4 — MC Greeks vs Analytical Greeks

Estimate **Delta** and **Vega** via Monte Carlo finite differences (bump-and-reprice)
using Common Random Numbers, then compare with the closed-form Black-Scholes values.

In [ ]:
N_GREEK = 50_000
cfg_g = SimulatorConfig(S0=BASE.S0, K=BASE.K, T=BASE.T, r=BASE.r,
                        sigma=BASE.sigma, n_paths=N_GREEK, n_steps=252, seed=None)
sim_g = GBMSimulator(cfg_g)
pay_g = VanillaCall(cfg_g.K)

greek_rows = []
for method_key, label in [('standard', 'Standard MC'), ('antithetic', 'Antithetic'),
                           ('sobol', 'Sobol QMC')]:
    g = mc_greeks(sim_g, pay_g, method=method_key)
    greek_rows.append({'Method': label, 'MC Delta': g['delta'], 'MC Vega': g['vega']})

greek_rows.append({'Method': 'BS Analytical',
                   'MC Delta': BS_GREEKS['delta'],
                   'MC Vega': BS_GREEKS['vega']})

df_greeks = pd.DataFrame(greek_rows).set_index('Method')
print(df_greeks.to_string(float_format='{:.6f}'.format))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)

method_labels = df_greeks.index.tolist()
greek_colors  = ['#1f77b4', '#ff7f0e', '#9467bd', '#d62728']

for ax, greek_col, title, bs_val in zip(
        axes,
        ['MC Delta', 'MC Vega'],
        ['Delta  Δ = ∂C/∂S₀', 'Vega  ν = ∂C/∂σ  (per 1% vol)'],
        [BS_GREEKS['delta'], BS_GREEKS['vega']]):

    vals = df_greeks[greek_col].values
    bars = ax.bar(method_labels, vals, color=greek_colors, alpha=0.85, edgecolor='white')
    ax.axhline(bs_val, color='red', ls='--', lw=1.5, label=f'BS = {bs_val:.4f}')
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Greek Value', fontsize=11)
    ax.legend(fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle(f'MC Greeks vs Black-Scholes Analytical  (N = {N_GREEK:,})', fontsize=13)
plt.tight_layout()
plt.savefig('../notebooks/plot_greeks_comparison.png', dpi=150)
plt.show()

---
## Section 5 — Variance Reduction & Efficiency Scorecard

Summary table showing:
- **Variance Reduction %** — how much variance is removed vs Standard MC
- **Efficiency Gain** — effective paths-per-actual-path (higher = better)
- **Paths to 1 % APE** — how many paths are needed to get within 1 % of BS

In [ ]:
N_SCORE = 50_000
cfg_s = SimulatorConfig(S0=BASE.S0, K=BASE.K, T=BASE.T, r=BASE.r,
                        sigma=BASE.sigma, n_paths=N_SCORE, n_steps=252, seed=None)
sim_s   = GBMSimulator(cfg_s)
pay_eur = VanillaCall(cfg_s.K)
pay_asi = AsianArithmetic(cfg_s.K)
bs_ctrl = black_scholes(cfg_s, 'call')

score_results = {
    'Standard MC':      sim_s.price(pay_eur, 'standard'),
    'Antithetic':       sim_s.price(pay_eur, 'antithetic'),
    'Control Variates': control_variate_price(sim_s, pay_asi, pay_eur, bs_ctrl),
    'Sobol QMC':        sim_s.price(pay_eur, 'sobol'),
}

std_se  = score_results['Standard MC']['std_error']
std_var = std_se ** 2

# Paths to 1% APE from earlier sweep
paths_1pct = {}
N_SWEEP_SCORE = [200, 500, 1_000, 2_000, 5_000, 10_000, 25_000, 50_000]
for m_label, m_key in [('Standard MC','standard'),('Antithetic','antithetic'),
                        ('Sobol QMC','sobol')]:
    paths_1pct[m_label] = None
    for n in N_SWEEP_SCORE:
        c = SimulatorConfig(S0=BASE.S0, K=BASE.K, T=BASE.T, r=BASE.r,
                            sigma=BASE.sigma, n_paths=n, n_steps=N_STEPS_SWEEP, seed=42)
        p = GBMSimulator(c).price(pay_eur if m_label!='Control Variates' else pay_asi,
                                   m_key)['price']
        if abs(p - BS_CALL) / BS_CALL * 100 < 1.0:
            paths_1pct[m_label] = n
            break

# CV paths to 1%
paths_1pct['Control Variates'] = None
for n in N_SWEEP_SCORE:
    c = SimulatorConfig(S0=BASE.S0, K=BASE.K, T=BASE.T, r=BASE.r,
                        sigma=BASE.sigma, n_paths=n, n_steps=N_STEPS_SWEEP, seed=42)
    sim_cv = GBMSimulator(c)
    cv_p = control_variate_price(sim_cv, AsianArithmetic(c.K), VanillaCall(c.K),
                                  black_scholes(c, 'call'))['price']
    if abs(cv_p - BS_CALL) / BS_CALL * 100 < 1.0:
        paths_1pct['Control Variates'] = n
        break

rows = []
for m, res in score_results.items():
    se  = res['std_error']
    var = se ** 2
    if m == 'Control Variates':
        vr_pct = res.get('variance_reduction_pct', 0.0)
    else:
        vr_pct = max(0.0, (1 - var / std_var) * 100) if std_var > 0 else 0.0
    eff    = std_var / var if var > 0 else float('inf')
    ci_w   = res['conf_interval'][1] - res['conf_interval'][0]
    ape    = abs(res['price'] - BS_CALL) / BS_CALL * 100
    p1pct  = paths_1pct.get(m)
    rows.append({
        'Method':            m,
        'Price':             f"${res['price']:.5f}",
        'APE vs BS (%)':     f"{ape:.3f}",
        'CI Width':          f"{ci_w:.5f}",
        'Var. Reduction %':  f"{vr_pct:.1f}",
        'Efficiency  ×std':  f"{eff:.2f}×",
        'Paths to 1% APE':   str(p1pct) if p1pct else f'>{ N_SWEEP_SCORE[-1]:,}',
    })

df_score = pd.DataFrame(rows).set_index('Method')
print(df_score.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
method_names = list(score_results.keys())
bar_colors   = ['#1f77b4','#ff7f0e','#2ca02c','#9467bd']

# Plot 1: APE
apes = [float(r.split('%')[0].split()[0]) if '%' not in r else float(r.replace('%',''))
        for r in df_score['APE vs BS (%)']]
apes = [abs(score_results[m]['price'] - BS_CALL) / BS_CALL * 100 for m in method_names]
axes[0].bar(method_names, apes, color=bar_colors, alpha=0.85, edgecolor='white')
axes[0].set_ylabel('APE vs BS (%)', fontsize=11)
axes[0].set_title('Accuracy', fontsize=12)
for i, v in enumerate(apes):
    axes[0].text(i, v + 0.002, f'{v:.3f}%', ha='center', va='bottom', fontsize=9)

# Plot 2: CI Width
ci_widths_score = [score_results[m]['conf_interval'][1] - score_results[m]['conf_interval'][0]
                   for m in method_names]
axes[1].bar(method_names, ci_widths_score, color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].set_ylabel('CI Width', fontsize=11)
axes[1].set_title('Precision (lower = better)', fontsize=12)
for i, v in enumerate(ci_widths_score):
    axes[1].text(i, v + 0.0001, f'{v:.4f}', ha='center', va='bottom', fontsize=9)

# Plot 3: Variance Reduction %
vr_list = []
for m, res in score_results.items():
    if m == 'Control Variates':
        vr_list.append(res.get('variance_reduction_pct', 0.0))
    else:
        var = res['std_error'] ** 2
        vr_list.append(max(0.0, (1 - var / std_var) * 100) if std_var > 0 else 0.0)
axes[2].bar(method_names, vr_list, color=bar_colors, alpha=0.85, edgecolor='white')
axes[2].set_ylabel('Variance Reduction (%)', fontsize=11)
axes[2].set_title('Variance Reduction vs Standard MC', fontsize=12)
for i, v in enumerate(vr_list):
    axes[2].text(i, v + 0.5, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

for ax in axes:
    ax.tick_params(axis='x', labelrotation=12)

plt.suptitle(f'Efficiency Scorecard  (N = {N_SCORE:,})', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../notebooks/plot_scorecard.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n', df_score.to_string())